# 05 - Model Training (3-Layer Pipeline)

Loads the Layer 1 selector and final feature list produced by notebook 04, re-applies Layer 2 feature engineering to the full training set, filters to the final feature list, tunes XGBoost with Optuna, trains the final model, and saves the model artifact.

In [2]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import joblib
import json
import config
from src.io import logger
from src.models import build_model, xgb_safe_frame
from src.optimization import optimize_model
from src.feature_selection import create_extended_engineered_features
from sklearn.metrics import roc_auc_score

ImportError: Please install optuna: pip install optuna

## Step 1: Load Preprocessed Training Data

In [ ]:
X_train = pd.read_csv(config.PROCESSED_DIR / "X_train_preprocessed.csv", index_col=0)

y_train_df = pd.read_csv(config.PROCESSED_DIR / "y_train.csv")
y_train = y_train_df.iloc[:, 0] if len(y_train_df.columns) == 1 else y_train_df["BCR"]

logger.info(f"Training data loaded: {X_train.shape}, Positives: {int(y_train.sum())}")

## Step 2: Load Artifacts From Notebook 04

Requires `fitted_layer1_selector.joblib` and `selected_features_final.csv`. Raises a clear error if notebook 04 has not been run.

In [ ]:
selector_path = config.MODELS_DIR / "fitted_layer1_selector.joblib"
features_path = config.TABLES_DIR / "selected_features_final.csv"

try:
    fitted_l1 = joblib.load(selector_path)
    selected_df = pd.read_csv(features_path)
    final_features = selected_df["feature"].tolist()
    logger.info(f"Loaded {len(final_features)} final features from the 3-layer pipeline.")
except FileNotFoundError as e:
    raise FileNotFoundError(
        f"{e}\n\n"
        "Please run '04_feature_selection.ipynb' first to generate "
        "'fitted_layer1_selector.joblib' and 'selected_features_final.csv'."
    )

## Step 3: Apply Layer 2 Feature Engineering and Filter to Final Features

Engineering is applied to the full training set before filtering, matching how notebook 04 built the candidate pool. Any final feature missing after engineering (should not normally happen, but can for edge splits) is filled with 0.0.

In [ ]:
X_train_eng, _ = create_extended_engineered_features(
    X_train, selected_genes=fitted_l1["mi_features"]
)

available_in_train = [f for f in final_features if f in X_train_eng.columns]
X_train_final = X_train_eng[available_in_train].copy()

missing_feats = set(final_features) - set(available_in_train)
if missing_feats:
    logger.warning(f"Missing {len(missing_feats)} features in training data: {list(missing_feats)[:5]}...")
    for feat in missing_feats:
        X_train_final[feat] = 0.0
    X_train_final = X_train_final[final_features]

logger.info(f"Final training matrix shape: {X_train_final.shape}")

## Step 4: Hyperparameter Tuning with Optuna

In [ ]:
logger.info("Starting Optuna hyperparameter tuning for XGBoost...")
n_trials = config.N_OPTUNA_TRIALS if hasattr(config, "N_OPTUNA_TRIALS") else 100
_, study, best_params = optimize_model(
    model_name="XGBoost",
    X_train=X_train_final,
    y_train=y_train,
    n_trials=n_trials,
    cv_splits=config.INNER_SPLITS,
    scoring="roc_auc",
)

logger.info(f"Best CV AUC: {study.best_value:.4f}")
logger.info(f"Best params: {json.dumps(best_params, indent=2, default=str)}")

pd.DataFrame([best_params]).to_csv(config.TABLES_DIR / "best_hyperparameters.csv", index=False)

## Step 5: Train Final Model on Full Training Set

In [ ]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
best_params["scale_pos_weight"] = n_neg / max(n_pos, 1)
best_params["random_state"] = config.RANDOM_STATE
best_params["n_jobs"] = -1
best_params["eval_metric"] = "logloss"

final_model = build_model("XGBoost", **best_params)
X_train_safe = xgb_safe_frame(X_train_final)
final_model.fit(X_train_safe, y_train)

train_pred = final_model.predict_proba(X_train_safe)[:, 1]
train_auc = roc_auc_score(y_train, train_pred)
logger.info(f"Training AUC (sanity check): {train_auc:.4f}")

## Step 6: Save Artifacts

In [ ]:
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, config.MODELS_DIR / "best_model_xgboost.joblib")
joblib.dump(fitted_l1, config.MODELS_DIR / "fitted_layer1_selector.joblib")

print("Model training complete. Artifacts saved:")
print(f"  - {config.MODELS_DIR / 'best_model_xgboost.joblib'}")
print(f"  - {config.MODELS_DIR / 'fitted_layer1_selector.joblib'}")